# Bank Distress Early-Warning Model — Modeling
**MSDS 696 Data Science Practicum II · Oussama Ennaciri**

In [3]:
from pathlib import Path

import numpy as np
import pandas as pd

PROCESSED = Path("..") / "data" / "processed"

panel = pd.read_parquet(PROCESSED / "panel_trend.parquet")
print(f"panel_trend: {panel.shape[0]:,} rows x {panel.shape[1]} cols")

panel_clean: 1,258,888 rows x 66 cols


### Capital headroom — the untrained benchmark

The rule says a bank must hold at least 8% total risk-based capital (and meet three other
minimums). A bank at 12% has four points of room; a bank at 8.5% has half a point. Less
room means closer to crossing the line.

**Headroom = the smallest gap between any of a bank's capital ratios and its minimum.**
The smallest gap is used because that is how the regulation itself works — the worst ratio
sets the capital tier, so it is the binding constraint.

The applicable ratios change in 2015 (Basel III adds common equity tier 1 and raises the
Tier 1 minimum), so the thresholds switch with the date, matching the label built in
`feature_selection.ipynb`. Ratios that do not apply in a given era contribute nothing.

It is built here, before the split, so the benchmark travels with the test rows. It is a
benchmark and not a feature, so the guardrail below excludes it from the model inputs.

In [ ]:
# Minimums for "adequately capitalized" — below any of these is undercapitalized.
# Source: 12 CFR 325.103 (1990-2014) and 12 CFR 324.403 (2015+), per
# ../literature/pca_label_definition.md
THRESHOLDS = {
    "pre2015": {"RBCRWAJ": 8.0, "RBC1RWAJ": 4.0, "RBC1AAJ": 4.0},
    "basel3":  {"RBCRWAJ": 8.0, "RBC1RWAJ": 6.0, "RBC1AAJ": 4.0, "RBCT1CER": 4.5},
}
BASEL3_START = pd.Timestamp("2015-01-01")

pre_basel3 = panel["REPDTE"] < BASEL3_START
gaps = pd.DataFrame(index=panel.index)

for ratio in ["RBCRWAJ", "RBC1RWAJ", "RBC1AAJ", "RBCT1CER"]:
    # A ratio absent from a regime's dict gets a NaN threshold, so its gap is NaN and
    # min() skips it -- e.g. CET1 simply does not exist before 2015.
    limit = np.where(pre_basel3,
                     THRESHOLDS["pre2015"].get(ratio, np.nan),
                     THRESHOLDS["basel3"].get(ratio, np.nan))
    gaps[ratio] = panel[ratio] - limit

panel["headroom"] = gaps.min(axis=1)

print(f"headroom: {panel['headroom'].isna().mean() * 100:.2f}% missing")
print(panel["headroom"].describe(percentiles=[.05, .25, .5, .75]).round(2).to_string())

## Leakage guardrail

The target (`onset_4q`) asks whether a healthy bank falls to undercapitalized within the
**next four quarters**. That forward look is what makes this problem leak-prone: any column
or any split that lets a training row see past its own prediction quarter hands the model
the answer, and the score comes back beautiful and meaningless.

This section fixes the rules **once**, before a single model is fit:

1. which columns are allowed to be inputs,
2. how the data is cut by time,
3. assertions that fail loudly if either rule is broken.

Everything below this point can then use `FEATURES`, `train`, and `test` without
re-checking leakage each time.

### 1 · Columns that can never be inputs

Four groups have to come out of the feature list, for four different reasons:

| Group | Columns | Why it can't be an input |
|---|---|---|
| **Identifiers** | `CERT`, `REPDTE`, `NAME`, `ESTYMD` | Not predictors — a certificate number carries no risk information. Kept in the table for reference and for the time split |
| **Label parts** | `onset_4q`, `quarters_to_onset`, `pca_tier`, `is_distressed`, `is_healthy` | These *are* the answer, or what the answer was built from. `quarters_to_onset` literally records how soon distress arrived |
| **Retroactive macro** | `USREC` | The NBER recession flag is dated **after the fact** — the Dec 2007 recession start was announced Dec 2008. A `1` sitting in a 2008Q1 row is knowledge nobody had that quarter, arriving exactly when failures spike |
| **Future-built flags** | `near_merge_exit`, `fast_failure` | Both were built in `cleaning.ipynb` from outcomes only known afterward. `near_merge_exit` is positive **0.00%** of the time against a 0.74% baseline — a free "this bank is safe" signal |

Dropping `USREC` costs nothing: the other macro columns (rates, unemployment, credit
spreads, financial stress) carry the same business-cycle signal and *were* genuinely
published at the time.

**`roa_artifact` stays in.** It's computed from the current quarter's ROA only, so it looks
backward, not forward. It is the one flag that is safe as an input.

The two dropped flags are **not deleted** — they stay in `panel` so results can be reported
separately on the hard cases (`fast_failure`) and sensitivity checked both ways
(`near_merge_exit`), exactly as `cleaning.ipynb` intended.

In [5]:
# --- The drop list: one definition, used everywhere below -----------------------

# Identifiers and raw dates. Not predictors, but needed for the split and for reference.
ID_COLS = ["CERT", "REPDTE", "NAME", "ESTYMD"]

# The target and everything it was derived from.
LABEL_COLS = ["onset_4q", "quarters_to_onset", "pca_tier", "is_distressed", "is_healthy"]

# Flags built from information that only exists after the prediction quarter.
FUTURE_FLAGS = ["near_merge_exit", "fast_failure"]

# Macro series dated retroactively. NBER announces recession dates months to a year
# after they begin, so this column wasn't knowable in the quarter it marks.
LOOKAHEAD_MACRO = ["USREC"]

DROP_FROM_FEATURES = ID_COLS + LABEL_COLS + FUTURE_FLAGS + LOOKAHEAD_MACRO

TARGET = "onset_4q"
FEATURES = [c for c in panel.columns if c not in DROP_FROM_FEATURES]

# Text columns that will need encoding before any model that can't take strings.
CATEGORICAL = [c for c in FEATURES if str(panel[c].dtype) in ("object", "category")]

print(f"features kept:   {len(FEATURES)}")
print(f"columns dropped: {len(DROP_FROM_FEATURES)}  ->  {DROP_FROM_FEATURES}")
print(f"needs encoding:  {CATEGORICAL}")
print(f"roa_artifact kept as a feature: {'roa_artifact' in FEATURES}")

features kept:   55
columns dropped: 11  ->  ['CERT', 'REPDTE', 'NAME', 'ESTYMD', 'onset_4q', 'quarters_to_onset', 'pca_tier', 'is_distressed', 'is_healthy', 'near_merge_exit', 'fast_failure']
needs encoding:  []
roa_artifact kept as a feature: True


### 2 · Split by time, with a gap year

**Why not a random split.** Shuffling rows puts 2008 and 2023 on both sides of the line.
The model trains on the same crisis it is later tested on and scores near-perfectly on
nothing at all.

**Why a gap year.** A training row at 2015Q4 is labeled using outcomes through 2016Q4.
If the test period opened in 2016, that training label would already describe the test
period. The gap is set to a full **four quarters** — exactly the label's horizon — so
every training label resolves before the test window opens.

**Why the test set stops at 2025Q1.** The panel runs to 2026Q1, but a row needs four
following quarters to be labeled at all. From 2025Q2 on, `cleaning.ipynb` blanked the
unobservable negatives (concern #8) and kept only the positives that had already crossed —
leaving **12 rows that are 100% positive with no negatives at all**. Left in the test set
they quietly inflate every recall figure. 2025Q1 is the last fully observable quarter.

Those trailing quarters aren't wasted: they keep every predictor, so they become the
**"who looks risky right now"** scoring set for the final presentation — a demo output,
not a validated score.

| Window | Dates | Purpose |
|---|---|---|
| **Train** | 1990Q1 – 2015Q4 | Fit the model |
| **Gap** | 2016Q1 – 2016Q4 | Buffer. Never fit on, never scored |
| **Test** | 2017Q1 – 2025Q1 | Honest evaluation, includes the 2023 failures |
| **Score now** | 2025Q2 – 2026Q1 | Unlabelable. Demo predictions only |

In [6]:
# --- Time boundaries ------------------------------------------------------------

TRAIN_END  = "2015-12-31"   # last quarter used for fitting
GAP_END    = "2016-12-31"   # 4 quarters wide = the label's forward horizon
TEST_START = "2017-01-01"
TEST_END   = "2025-03-31"   # last quarter with a full 4-quarter outcome window

# Only rows that carry a real label can be trained or scored on. Blank onset_4q means
# either "already distressed" or "outcome window not observable" — neither is trainable.
labeled = panel[panel[TARGET].notna()]

train = labeled[labeled["REPDTE"] <= TRAIN_END]
test  = labeled[(labeled["REPDTE"] >= TEST_START) & (labeled["REPDTE"] <= TEST_END)]

# Held out on purpose. Exists to be excluded, not used.
gap = labeled[(labeled["REPDTE"] > TRAIN_END) & (labeled["REPDTE"] <= GAP_END)]

# No label yet, but every predictor is present -> the live "who looks risky now" set.
score_now = panel[(panel["REPDTE"] > TEST_END) & (panel[TARGET].isna())]

for name, df in [("train", train), ("gap", gap), ("test", test)]:
    print(f"{name:<6} {len(df):>10,} rows   "
          f"{df['REPDTE'].min().date()} to {df['REPDTE'].max().date()}   "
          f"{int(df[TARGET].sum()):>5,} positives ({df[TARGET].mean() * 100:.2f}%)")

print(f"{'score':<6} {len(score_now):>10,} rows   "
      f"{score_now['REPDTE'].min().date()} to {score_now['REPDTE'].max().date()}   "
      f"unlabeled (demo only)")

train   1,026,767 rows   1990-03-31 to 2015-12-31   8,404 positives (0.82%)
gap        24,180 rows   2016-03-31 to 2016-12-31      31 positives (0.13%)
test      166,329 rows   2017-03-31 to 2025-03-31     563 positives (0.34%)
score      17,697 rows   2025-06-30 to 2026-03-31   unlabeled (demo only)


### 3 · Assertions

The rules above are only worth something if breaking them is noisy. Each check below
corresponds to a specific way this project could leak:

1. **No leaky column survived** — catches a future-built flag sliding back into `FEATURES`
   after an edit upstream.
2. **Train ends before test begins** — catches an accidental shuffle or a bad date string.
3. **Training labels resolve before the test window** — the gap-year check, stated in terms
   of the label horizon rather than trusting the dates by eye.
4. **The rare-event rate survives in both halves** — catches rebalancing (`SMOTE`, class
   weights) applied before the split instead of to the training fold only.
5. **Every test quarter contains real negatives** — catches the positives-only tail from
   concern #8, and any future version of the same mistake.

If a cell below this one ever changes the splits, re-run this cell.

In [7]:
# --- Guardrail checks: each one fails loudly on a specific leak -------------------

# 1 · No dropped column made it into the feature list.
leaked = set(DROP_FROM_FEATURES) & set(FEATURES)
assert not leaked, f"leaky columns in FEATURES: {leaked}"

# 2 · Train and test never overlap in time.
assert train["REPDTE"].max() < test["REPDTE"].min(), "train and test windows overlap"

# 3 · Every training label resolves before the test window opens.
#     A row at quarter t is labeled from t+1..t+4, so push the last training date
#     forward by 4 quarters and check it still lands before the test starts.
last_train_label_resolves = train["REPDTE"].max() + pd.DateOffset(months=12)
assert last_train_label_resolves < test["REPDTE"].min(), (
    f"training labels resolve at {last_train_label_resolves.date()}, "
    f"inside the test window starting {test['REPDTE'].min().date()}"
)

# 4 · Both halves keep the true rare-event rate — no rebalancing has happened yet.
for name, df in [("train", train), ("test", test)]:
    rate = df[TARGET].mean()
    assert 0.001 < rate < 0.05, f"{name} positive rate {rate:.4f} is not the real base rate"

# 5 · Every test quarter has real negatives (guards the positives-only tail, concern #8).
quarters_without_negatives = [
    d.date() for d, s in test.groupby("REPDTE")[TARGET] if not (s == 0).any()
]
assert not quarters_without_negatives, (
    f"test quarters with no negatives: {quarters_without_negatives}"
)

print("all leakage checks passed")
print(f"  train: {train['REPDTE'].min().date()} to {train['REPDTE'].max().date()}")
print(f"  test:  {test['REPDTE'].min().date()} to {test['REPDTE'].max().date()}")
print(f"  gap:   {gap['REPDTE'].min().date()} to {gap['REPDTE'].max().date()} (excluded)")

all leakage checks passed
  train: 1990-03-31 to 2015-12-31
  test:  2017-03-31 to 2025-03-31
  gap:   2016-03-31 to 2016-12-31 (excluded)


### What this does not cover

Three leakage risks live in code that doesn't exist yet. They get handled where they arise,
not here:

- **Trend features.** The funding signals still to build (deposit growth, uninsured-deposit
  share, held-to-maturity losses) must use backward-only windows. A rolling window centred on
  the prediction quarter, or any `shift(-1)`, reads the future.
- **Feature ranking.** The single-feature scores (AUC) in `eda.ipynb` were computed across
  all years, test period included. Re-rank on `train` only, then freeze the list.
- **Filling blanks and scaling.** Fit on `train`, then apply to `test` — never a statistic
  computed over both. Tree models (gradient boosting) take blanks natively and sidestep this.

One remaining limitation, not a fix: the economic columns (`FRED`) are treated as available
in their own quarter, but several publish late — GDP (`GDPC1`) about a month after the
quarter closes, house prices (`USSTHPI`) and lending standards (`DRTSCILM`) later still, all
revised afterward. Milder than `USREC`, since the lag is short and fixed rather than a
committee decision. Lag the macro block one quarter to be strict, or note it in the writeup.

## What is being compared: three benchmarks, four methods

A single comparison answers a single objection. Four reference points are used here because
four different objections get raised about a model like this.

**Benchmarks — nothing is fitted:**

| Benchmark | What it is | Objection it answers |
|---|---|---|
| **Naive floor** | Flag every bank | Is the model doing anything at all? |
| **Capital headroom** | Rank by how close the weakest capital ratio sits to its regulatory minimum | Does it beat what supervision already does? |
| **SCOR** | The FDIC's own off-site system, from the published record | Does it beat the system actually in production? |

**Methods — all fitted on the same training window and the same features:**

| Method | Role |
|---|---|
| **Gradient boosting** | Champion — the method being defended |
| **Logistic regression** | The alternative: simpler, explainable, and the thing the champion must beat |
| **MLP** | Neural network on the same tabular features (`deep_learning.ipynb`) |
| **GRU** | Sequence model reading eight quarters in order (`deep_learning.ipynb`) |

The naive floor exists because Correia/Luck/Verner (2024) report performance as a *multiple of*
what flagging everything achieves; a model that cannot beat it has found nothing. Capital
headroom is the operational status quo — Prompt Corrective Action *is* threshold monitoring.
SCOR is the only external number in the set that was measured on a comparable task.

By way of contrast, Carmona et al. (2018) — the one paper here using this project's champion
method — reports no baseline at all. Their three "models" are three tuning stages of the same
XGBoost.

In [ ]:
# The benchmark score is derived from the capital ratios, which are already features.
# Leaving it in would hand the models the benchmark's own answer.
BENCHMARK_COLS = ["headroom"]

# Text columns are held back for this first run so both models see identical inputs.
# One-hot encoding state (56 levels) would give gradient boosting a different feature
# space than logistic regression, confounding the comparison. Noted as a follow-up.
FEATURES = [
    c for c in panel.columns
    if c not in DROP_FROM_FEATURES + BENCHMARK_COLS
    and str(panel[c].dtype) not in ("object", "category")
]

print(f"model features: {len(FEATURES)}")
print(f"  of which trend: {sum(1 for c in FEATURES if '_chg' in c or '_grow' in c)}")
print(f"held back (text, for later): {CATEGORICAL}")

## How performance is measured

Accuracy is useless here. Predicting "no distress" for every bank scores **99.66%** and
catches nothing. None of the ten papers reviewed uses it.

Four numbers instead, each answering a different question:

| Metric | Question it answers |
|---|---|
| **ROC-AUC** | Given one distressed and one healthy bank, how often is the distressed one ranked higher? 0.5 is a coin flip |
| **PR-AUC** | Of the banks flagged, what share are genuinely distressed? Sensitive to rarity, unlike ROC-AUC |
| **Lift** | PR-AUC divided by the base rate — how many times better than flagging everything. Correia's presentation |
| **Recall @ 1%** | With capacity to examine only the riskiest 1% of banks, what share of real cases are caught? |

Recall at a fixed alert budget is the one a risk committee actually asks about, and it
matches the asymmetric-cost framing in Cole & White: missing a failure costs far more than
a false alarm, but examiner time is finite.

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

ALERT_BUDGET = 0.01  # examine the riskiest 1% of banks


def evaluate(name, scores, y, budget=ALERT_BUDGET):
    """Score one ranking. Higher `scores` must mean higher risk."""
    scores = np.asarray(scores, dtype=float)
    n_alerts = int(len(scores) * budget)
    flagged = np.argsort(-scores)[:n_alerts]

    pr_auc = average_precision_score(y, scores)
    return {
        "model": name,
        "ROC-AUC": roc_auc_score(y, scores),
        "PR-AUC": pr_auc,
        "lift": pr_auc / y.mean(),          # multiple of the naive floor
        "recall@1%": y[flagged].sum() / y.sum(),
    }

## Fitting the two models

**Logistic regression** cannot handle blanks or wildly different scales, so it gets a
pipeline: fill blanks with the median, then standardise. Both steps are fitted on the
**training data only** and then applied to test — fitting them on everything would leak
test-period information into training, the preprocessing leak flagged at the end of this
notebook.

**Gradient boosting** takes blanks natively and is scale-invariant, so it gets the raw
columns. That is one of the reasons it suits this data, where the era gaps are structural.

Neither model is tuned beyond sensible defaults. Tuning until the champion wins is how a
comparison becomes an argument for a foregone conclusion.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X_train, y_train = train[FEATURES], train[TARGET].values
X_test,  y_test  = test[FEATURES],  test[TARGET].values

# Columns with no observed value at all in the training window cannot be imputed;
# drop them for the linear model only. (CET1 is empty before 2015.)
LR_FEATURES = [c for c in FEATURES if X_train[c].notna().any()]

logistic = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(max_iter=1000),
).fit(X_train[LR_FEATURES], y_train)

# class_weight="balanced" tells the model a missed case costs as much as the class is
# rare -- the rebalancing Petropoulos et al. (2020) achieve by downsampling instead.
# Applied to the training fit only; the test set keeps its true 0.34% rate.
boosting = HistGradientBoostingClassifier(
    max_iter=400,
    learning_rate=0.05,
    min_samples_leaf=50,
    l2_regularization=1.0,
    class_weight="balanced",
    random_state=0,
).fit(X_train, y_train)

print(f"logistic regression: {len(LR_FEATURES)} features")
print(f"gradient boosting:   {len(FEATURES)} features")

## Scoring rows whose outcome was never observed

The diagnostic above settles whether the merged-away banks contaminate the *label*. A separate
question is whether they should be *scored* at all.

If the model flags a bank and that bank is acquired before its four-quarter window closes,
nobody ever learns whether the warning was right. Counted as it stands, that is a false alarm —
the model is charged for an error that was never demonstrated. At a 1% alert budget, 125 of the
1,551 false positives are exactly this case, appearing at roughly 2.4 times their share of the
test set.

This is **censoring**, and survival analysis has a settled answer: an observation whose outcome
was not observed is excluded from scoring — neither a hit nor a miss. That is the treatment
adopted here, with the full test set reported underneath as a sensitivity check.

Note this is an evaluation decision, never a modelling one. At prediction time nobody knows a
bank is about to be acquired, so nothing about the merger can enter the features — that is
precisely the `near_merge_exit` leak the guardrail exists to block.

In [ ]:
# Censored: the outcome window never closed because the bank left the panel by merger.
test_scored = test[~test["near_merge_exit"]]
X_test_scored = test_scored[FEATURES]
y_test_scored = test_scored[TARGET].values

print(f"test rows            {len(test):,}")
print(f"censored (excluded)  {test['near_merge_exit'].sum():,}")
print(f"scored               {len(test_scored):,}  "
      f"({int(y_test_scored.sum())} cases, base rate {y_test_scored.mean():.4f})")

### Effect of the exclusion

It lifts PR-AUC by roughly 8% — and lifts the benchmark by the same proportion, because the
removed rows are hard negatives for every ranking, not just the model's. The miss rate does not
move at all, since every censored row is a negative and none was ever a case that could be
caught.

So the decision is about correctness, not advantage. Both versions appear in the results below.

## Results — test period 2017Q1 to 2025Q1

Primary figures use the scored set, with the censored rows removed as decided above. The full
test set is reported underneath so the effect of that decision is visible rather than assumed.

In [ ]:
headroom_score = -test["headroom"].fillna(test["headroom"].max()).values
headroom_scored = -test_scored["headroom"].fillna(test_scored["headroom"].max()).values

results = pd.DataFrame([
    evaluate("naive (flag everything)",
             np.random.RandomState(0).rand(len(y_test_scored)), y_test_scored),
    evaluate("capital headroom", headroom_scored, y_test_scored),
    evaluate("logistic regression",
             logistic.predict_proba(X_test_scored[LR_FEATURES])[:, 1], y_test_scored),
    evaluate("gradient boosting",
             boosting.predict_proba(X_test_scored)[:, 1], y_test_scored),
])

print(f"scored rows {len(test_scored):,} | cases {int(y_test_scored.sum()):,} "
      f"| base rate {y_test_scored.mean():.4f}\n")
print(results.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

sensitivity = pd.DataFrame([
    evaluate("capital headroom", headroom_score, y_test),
    evaluate("logistic regression",
             logistic.predict_proba(X_test[LR_FEATURES])[:, 1], y_test),
    evaluate("gradient boosting", boosting.predict_proba(X_test)[:, 1], y_test),
])
print(f"\n\nSENSITIVITY — all {len(test):,} test rows, censored counted as false alarms\n")
print(sensitivity.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

## Diagnostic — does it work when the failure mechanism matches?

The result above is weak, and the temptation is to tune until the champion wins. The more
useful question is *why* it is weak.

The training years (1990–2015) are dominated by credit-driven distress: bad loans, real
estate, slow deterioration. The 2017–2025 test period contains the 2021–23 wave, which was
driven by interest rates and deposit flight — a different mechanism. If that is the
explanation, the same model should perform well on a period whose failures *do* resemble
its training data.

So: train through 2005, skip 2006, and test on the 2008 crisis — which the model has never
seen, but which fails the same way its training years did.

In [ ]:
def run_split(train_end, test_start, test_end, label):
    """Refit everything on a different time split and return the comparison table."""
    tr = labeled[labeled["REPDTE"] <= train_end]
    te = labeled[(labeled["REPDTE"] >= test_start) & (labeled["REPDTE"] <= test_end)]
    y = te[TARGET].values

    lr_cols = [c for c in FEATURES if tr[c].notna().any()]
    lr = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                       LogisticRegression(max_iter=1000)).fit(tr[lr_cols], tr[TARGET])
    gb = HistGradientBoostingClassifier(max_iter=400, learning_rate=0.05,
                                        min_samples_leaf=50, l2_regularization=1.0,
                                        random_state=0).fit(tr[FEATURES], tr[TARGET])

    out = pd.DataFrame([
        evaluate("capital headroom", -te["headroom"].fillna(te["headroom"].max()).values, y),
        evaluate("logistic regression", lr.predict_proba(te[lr_cols])[:, 1], y),
        evaluate("gradient boosting", gb.predict_proba(te[FEATURES])[:, 1], y),
    ])
    print(f"\n{label}  |  train <= {train_end} ({len(tr):,})  ->  "
          f"test {test_start}..{test_end} ({len(te):,}, base {y.mean():.4f})")
    print(out.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
    return out


crisis_2008 = run_split("2005-12-31", "2007-01-01", "2011-12-31", "2008 CREDIT CRISIS")

## Reading the two tables together

Same model, same features, same code — two different eras, opposite verdicts.

On the 2008 crisis the champion wins clearly and by a wide margin. On 2017–2025 it barely
edges the untrained benchmark, and loses on ROC-AUC outright.

That is not a broken model. It is a model that learned one failure mechanism well and was
then tested on another. Petropoulos et al. (2020) found the same shape of result: logistic
regression held up out-of-sample but was worst in six of eight criteria out-of-time, once
the period changed.

Two honest conclusions:

1. **The method is sound** — given failures that resemble what it trained on, it beats both
   the simple model and the status quo.
2. **The training data is the limitation, not the algorithm.** The trend features added in
   `feature_engineering.ipynb` were the first step toward the 2023 mechanism. They were not
   enough on their own, and the next step is the funding-side signals that Chu et al. (2026)
   and the SVB case study both point to.

## Diagnostics — locating where the model actually fails

The headline table is one number covering nine years, which hides more than it shows. Two
sweeps break it apart. Both are **diagnostics, not model selection**: the reported model
above is fixed, and nothing below is used to choose it. That distinction matters, because
running many variants and reporting the winner would quietly turn the test set into a
training set.

### Sweep 1 — does older training data help or hurt?

The training window runs from 1990, but the 1990s and the 2000s were different banking
industries under different capital rules. If the early years are teaching the wrong lesson,
cutting them should improve performance on a modern test period.

Nine training start years, one fixed test period.

In [ ]:
def train_from(start_year):
    """Refit the champion on a shortened training window. Test period never changes."""
    tr = train[train["REPDTE"] >= f"{start_year}-01-01"]
    m = HistGradientBoostingClassifier(
        max_iter=400, learning_rate=0.05, min_samples_leaf=50,
        l2_regularization=1.0, class_weight="balanced", random_state=0,
    ).fit(tr[FEATURES], tr[TARGET])
    row = evaluate(str(start_year), m.predict_proba(X_test)[:, 1], y_test)
    row["train_rows"], row["train_pos"] = len(tr), int(tr[TARGET].sum())
    return row


sweep_era = pd.DataFrame([train_from(y) for y in
                          [1990, 1993, 1996, 1999, 2002, 2005, 2008, 2010, 2012]])
sweep_era = sweep_era.rename(columns={"model": "train from"})
print(sweep_era[["train from", "train_rows", "train_pos",
                 "ROC-AUC", "PR-AUC", "recall@1%"]].to_string(
      index=False, float_format=lambda v: f"{v:.4f}"))

### Reading Sweep 1: the era does not matter much, and 2010 is a mirage

Everything from 1993 to 2005 lands in a narrow band around 0.070 PR-AUC. There is no cliff,
so there is no regime break hiding in the training data.

The one apparent standout, 2010, is worth looking at closely, because it is exactly the kind
of result that gets reported without scrutiny:

| Train from | PR-AUC |
|---|---|
| 2008 | 0.063 |
| **2010** | **0.078** |
| 2012 | 0.052 |

A genuine effect would trend. A single high point wedged between two low neighbours is
sampling noise — and by 2012 the training set holds only 370 positives, so the estimates
there are unstable to begin with. Cutting to 2010 would also discard the 2008 crisis, the
densest source of real distress cases in the data.

**Decision: keep the full 1990–2015 window.** Not because it scored best, but because the
sweep shows nothing is gained by cutting, and cutting invites a choice made on test results.

### Sweep 2 — when does the model stop working?

The same fixed model, scored one test year at a time. If performance degrades smoothly, the
problem is drift. If it falls off a cliff in specific years, something changed in those years.

Base rates differ year to year, so PR-AUC is also shown as **lift** — the multiple of what
flagging at random would achieve — which is comparable across rows.

In [ ]:
scored = test.copy()
scored["model_score"] = boosting.predict_proba(X_test)[:, 1]
scored["bench_score"] = -scored["headroom"].fillna(scored["headroom"].max())

rows = []
for year, block in scored.groupby(scored["REPDTE"].dt.year):
    y = block[TARGET].values
    if y.sum() < 3:                      # too few cases for a stable estimate
        continue
    model_pr = average_precision_score(y, block["model_score"])
    bench_pr = average_precision_score(y, block["bench_score"])
    rows.append({
        "year": year, "banks": len(block), "cases": int(y.sum()),
        "base rate": y.mean(),
        "ROC-AUC": roc_auc_score(y, block["model_score"]),
        "model lift": model_pr / y.mean(),
        "bench lift": bench_pr / y.mean(),
    })

sweep_year = pd.DataFrame(rows)
print(sweep_year.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

hard = sweep_year[sweep_year["year"].between(2021, 2023)]["cases"].sum()
print(f"\n2021-2023 holds {hard} of {int(y_test.sum())} test cases "
      f"({hard / y_test.sum() * 100:.0f}%)")

### Reading Sweep 2: the failure has a date

The model does not degrade. It works, stops working for three years, and starts working
again:

| Period | ROC-AUC | Verdict |
|---|---|---|
| 2017–2019 | 0.95–0.97 | Strong. Beats the benchmark roughly 2:1 in 2018 and 2019 |
| **2021–2023** | **0.53–0.75** | **Collapse.** 2021 is barely better than a coin flip |
| 2024–2025 | 0.81–0.92 | Recovered |

And the collapse lands exactly where the cases are: **2021–2023 holds 435 of the 563 test
cases, 77% of them.** That is why the headline number looked mediocre — the aggregate is
dominated by the one window the model cannot read.

This is the same shape of result Petropoulos et al. (2020) report: a model that validates
well within its own period and degrades once the period changes.

**What changed in those years.** The training data ends in 2015 and is dominated by
credit-driven distress — bad loans, real estate, gradual deterioration over two years, the
pattern in the EDA. The 2021–23 wave was driven by interest-rate shock and deposit flight:
fast, funding-side, and with capital ratios that *rose* into failure (the SVB case study in
`eda.ipynb`). The model was not taught that mechanism, and the funding features added in
`feature_engineering.ipynb` are a first step at it, not a solution.

**What this changes about the claim.** Not "gradient boosting beats the status quo" — it
does not, on average, over this test period. The defensible claim is narrower and more
useful: *the method works on the failure mechanism it was trained on, and the 2021–23 window
is a different mechanism, which is measurable, dateable, and the specific thing the next
round of features has to address.*

## Context: how well does the FDIC's own system do?

Every comparison above is against something built in this notebook. It is worth knowing what a
real off-site monitoring system achieves — not to compete with it, but to calibrate what
"good" looks like on this kind of problem.

**SCOR** is the FDIC's off-site system. It scores every bank's call report between examinations
and flags those likely to be downgraded at the next one. Collier et al. (2003) publish its
year-by-year record, 1986–2002:

| | SCOR, 1986–2002 |
|---|---|
| Bank-examinations | 126,505 |
| Actual downgrades | 7,162 (5.66% base rate) |
| Flagged | 8,622 (6.8% of those examined) |
| Correct | 3,196 |
| **Caught** | **44.6%** |
| **False alarms** | **62.9%** |

**This is not a benchmark for this project, and it is deliberately not in the results table.**
SCOR predicts a *CAMELS downgrade* — an examiner's judgment, applied to banks already scheduled
for examination, at a four-to-six-month horizon. This project predicts a *capital ratio crossing
a statutory threshold*, across all healthy banks, four quarters out. The events differ, the
populations differ, and the base rates differ by a factor of sixteen. Scoring them side by side
would be comparing two different problems.

What it does establish is the difficulty of the task. **The FDIC, using its own supervisory data
on a denser and more forgiving target, catches fewer than half the cases it is looking for.**
Any claim that a model built from public call reports should be catching most of them is not
supported by the operational record.

Two further details worth carrying into the writeup:

- **The widely-quoted "about two-thirds missed" is loose.** Over the full record it is 55.4%,
  though it does exceed two-thirds after 1993.
- **SCOR degraded the same way this project's model does.** Its miss rate rose from 27–52% in
  1986–1991 to 76–89% across 1993–2002 — built on the banking crisis of the late 1980s, it lost
  most of its power once the failure mechanism changed. The regime sensitivity found in Sweep 2
  is a known property of call-report monitoring, visible in the FDIC's own production system.

In [ ]:
# SCOR's published record (Collier et al. 2003, Table 4), for context only.
SCOR = {"examined": 126_505, "downgraded": 7_162, "flagged": 8_622, "correct": 3_196}
print(f"SCOR, 1986-2002: caught {SCOR['correct'] / SCOR['downgraded']:.1%} of downgrades "
      f"at a {SCOR['flagged'] / SCOR['examined']:.1%} alert rate\n")

# Any single alert budget is arbitrary -- it depends on examiner capacity, not on the
# model. Cole & White (2012) report error rates across a range for the same reason.
BUDGETS = (0.005, 0.01, 0.02, 0.05)


def error_profile(name, scores, y, budgets=BUDGETS):
    """Type I and Type II across alert budgets."""
    order = np.argsort(-np.asarray(scores, dtype=float))
    out = []
    for b in budgets:
        k = int(len(order) * b)
        caught = y[order[:k]].sum()
        out.append({"model": name, "alert budget": f"{b:.1%}", "banks flagged": k,
                    "caught": caught / y.sum(),
                    "Type I (missed)": 1 - caught / y.sum(),
                    "Type II (false alarms)": 1 - caught / k})
    return out


profiles = pd.DataFrame(
    error_profile("capital headroom", headroom_scored, y_test_scored)
    + error_profile("logistic regression",
                    logistic.predict_proba(X_test_scored[LR_FEATURES])[:, 1], y_test_scored)
    + error_profile("gradient boosting",
                    boosting.predict_proba(X_test_scored)[:, 1], y_test_scored)
)
print(profiles.to_string(index=False, float_format=lambda v: f"{v:.1%}"))

### Reading the ladder

There is no correct alert budget. It is set by how many banks supervisors can actually examine,
which is a resourcing question, not a modelling one — so performance is reported across a range
rather than at one point, following Cole & White (2012).

Two things the ladder makes visible that a single number hides:

- **Type I and Type II move in opposite directions.** Flagging more banks catches more cases and
  wastes more examiner time. Nothing in the model changes; only the willingness to look does.
- **False-alarm rates stay high at every budget, and that is arithmetic rather than weakness.**
  With 563 cases in 160,712 bank-quarters, flagging 1% means issuing 1,607 alerts for at most
  563 possible hits — so even a perfect model would be wrong two-thirds of the time. The figure
  to judge is the catch rate at a budget, not the false-alarm rate in isolation.

Where a single figure is needed, **1%** is used in the summary below. It is a round, tight
screen and nothing more — a reporting convention, not a derived result.

## Do the macro columns earn their place?

The panel joins eleven FRED series — rates, unemployment, GDP, house prices, credit spreads,
financial stress. That join was made on the assumption it was literature-backed. Re-reading
the source showed the opposite: Nuxoll (2003) ran exactly this test and concluded that
*"economic data do not improve these forecasts despite the fact that the data are
statistically significant."* Oshinsky & Olin (2005) excluded economic variables by choice.

So the join needs its own evidence. The test is an ablation — the same model, fitted twice,
with and without the macro block — which is the standard move in this literature (Nuxoll runs
it for economic data, Curry et al. for market data).

Run on both test periods, since a variable that is useless in calm years might still matter
in a crisis.

In [ ]:
MACRO = ["FEDFUNDS", "DGS10", "T10Y3M", "UNRATE", "GDPC1", "CPIAUCSL",
         "USSTHPI", "BAA10Y", "DRTSCILM", "NFCI", "BOGZ1FL075035503Q"]
WITHOUT_MACRO = [c for c in FEATURES if c not in MACRO]


def ablate(features, tr, te, label):
    m = HistGradientBoostingClassifier(
        max_iter=400, learning_rate=0.05, min_samples_leaf=50,
        l2_regularization=1.0, class_weight="balanced", random_state=0,
    ).fit(tr[features], tr[TARGET])
    return evaluate(label, m.predict_proba(te[features])[:, 1], te[TARGET].values)


train_2005 = labeled[labeled["REPDTE"] <= "2005-12-31"]
test_2008 = labeled[(labeled["REPDTE"] >= "2007-01-01") & (labeled["REPDTE"] <= "2011-12-31")]

ablation = pd.DataFrame([
    ablate(FEATURES,      train,      test,      "2017-2025  with macro"),
    ablate(WITHOUT_MACRO, train,      test,      "2017-2025  without macro"),
    ablate(FEATURES,      train_2005, test_2008, "2008 crisis  with macro"),
    ablate(WITHOUT_MACRO, train_2005, test_2008, "2008 crisis  without macro"),
])

print(f"{len(FEATURES)} features with macro, {len(WITHOUT_MACRO)} without\n")
print(ablation.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

### Verdict: Nuxoll was right, and still is

On the main test period the two models are **identical to four decimal places** — same PR-AUC,
same ROC-AUC to within 0.0003. Eleven features, no measurable contribution.

On the 2008 crisis the macro block adds about 4% of PR-AUC. Small, but the sign is consistent
with the idea that aggregate conditions matter when the whole system is moving together.

**Why this happens.** The macro columns hold the same eleven numbers for every bank in a given
quarter. They can raise or lower the predicted risk of the entire population at once, but they
carry no information about *which* bank is the problem — and picking which bank is the entire
task. That is Nuxoll's own explanation, and it holds on data running twenty-three years past
his.

**Decision: keep the block, and report this.** It costs nothing, it helps marginally in a
crisis, and the ablation is worth more in the writeup than the features are in the model — an
assumption inherited from a misread citation, tested directly, and resolved against a
published result.

The related caveat from the guardrail section still stands: several of these series publish
weeks after the quarter they describe, so a strict version would lag the whole block by one
quarter. Given they contribute nothing measurable, that refinement is not worth the complexity.

## Is the label contaminated by banks that merged away?

Concern #3 in `data_concerns.md`: when a bank's charter ends by merger, it leaves the panel,
so its last four quarters were labelled "stayed safe" without anyone observing whether it
would have. Those rows carry the `near_merge_exit` flag — 48,239 of them, about 4% of the
healthy negatives.

The worry is not academic. A weak bank often escapes by being acquired, and the Week 1
proposal lists voluntary merger as one of the corrective actions available to a distressed
bank. If that is common, those rows are distress cases sitting in the data labelled safe, and
every score above is understated.

Oshinsky & Olin (2005) handle this by making merger its own outcome in a four-way model. That
is a redesign. The cheaper first question is whether there is anything to redesign *for*: do
these banks look like the distressed ones, or like the survivors?

In [ ]:
VITALS = ["RBCRWAJ", "NPERFV", "ROA", "ROE", "EEFFR", "ORER"]
TRENDS = ["RBCRWAJ_chg8q", "NPERFV_chg8q", "ROA_chg8q", "EQV_chg8q"]

negatives = labeled[labeled[TARGET] == 0]
merged     = negatives[negatives["near_merge_exit"]]
survivors  = negatives[~negatives["near_merge_exit"]]
distressed = labeled[labeled[TARGET] == 1]

groups = {"merged away": merged, "healthy survivors": survivors,
          "became undercapitalized": distressed}

print("LEVELS — median vitals\n")
print(pd.DataFrame({k: v[VITALS].median() for k, v in groups.items()}).round(2).to_string())

print("\n\nTRENDS — median change over 8 quarters\n")
print(pd.DataFrame({k: v[TRENDS].median() for k, v in groups.items()}).round(3).to_string())

# What share of each group would a crude weakness screen pick up?
weak_capital = survivors["RBCRWAJ"].quantile(0.10)
weak_npa = survivors["NPERFV"].quantile(0.90)
print(f"\n\nSHARE LOOKING WEAK  (capital < {weak_capital:.2f}% or bad loans > {weak_npa:.2f}%)\n")
for name, block in groups.items():
    share = ((block["RBCRWAJ"] < weak_capital) | (block["NPERFV"] > weak_npa)).mean()
    print(f"  {name:<26} {share:6.1%}   (n = {len(block):,})")

print("\n\nSENSITIVITY — does removing them move the target rate?\n")
print(f"  all rows          {labeled[TARGET].mean():.3%}")
print(f"  merged excluded   {labeled[~labeled['near_merge_exit']][TARGET].mean():.3%}")

### Verdict: normal consolidation, and the label stands

The merged-away banks track the survivors on every measure, and are nowhere near the
distressed profile:

| Median | Merged away | Survivors | Became undercapitalized |
|---|---|---|---|
| Capital ratio | 14.61 | 15.76 | **10.19** |
| Bad loans | 0.59 | 0.58 | **4.03** |
| ROA | 0.91 | 1.01 | **−0.26** |
| Capital, 2-yr change | **+0.21** | +0.12 | **−1.49** |

The trend row is the decisive one. Banks heading for distress lose 1.49 points of capital
over two years — the decline the EDA chart is built on. Banks that merged away were *gaining*
capital, slightly faster than the survivors. Whatever drove those mergers, it was not the
deterioration this project is trying to catch.

The weakness screen agrees: 21.9% of merged banks trip it, against 18.1% of survivors and
81.9% of genuine cases. There is a mild tilt — some weak banks clearly are bought — but it is
a few points, not a hidden population.

**Decision: keep the rows, keep the label, drop the concern.** Removing all 48,239 shifts the
target rate from 0.740% to 0.771%, which changes nothing that matters. The four-way
multinomial redesign is not warranted: it would spend the project's remaining time modelling
an outcome that turns out to be ordinary industry consolidation, 69% of the panel by count.

This closes concern #3. The `near_merge_exit` flag stays in the table as the evidence for the
check, not as a correction to be applied.

---

# Deep learning challengers

Two neural networks on the same split, the same censored-row exclusion, and the same
metrics as everything above.

| Model | What it sees |
|---|---|
| **MLP** (multi-layer perceptron) | One quarter per bank — the same features gradient boosting gets |
| **GRU** (gated recurrent unit) | Eight quarters in order, so it reads the *shape* of a decline rather than being handed "capital fell 2 points" |

The GRU is the interesting one. This project's central EDA finding is that banks decline
over two years before crossing into distress, and a sequence model is the natural way to
learn that shape. Whether it beats features that hand the trend over directly is an open
question — and the answer below is no.

Petropoulos et al. (2020) tested neural networks on this problem and found them second to
tree ensembles, so there is precedent either way.

## Setup

One structural difference from the models above: early stopping needs a validation set, and it
must not be the test period. It is carved from the **end of training** — 2011–2015, which holds
about 670 cases. Stopping on 2013+ alone was tried first and abandoned; 220 cases gives a signal
too noisy to stop on.

So this section uses train ≤ 2010, validate 2011–2015, and the same test period as everything
else. Its variables are prefixed `dl_` so the splits used earlier stay untouched.

> torch and LightGBM each bundle their own OpenMP runtime, and importing both in one process
> aborts. The guard below must run before torch loads.

In [ ]:
import os

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch
import torch.nn as nn

DEV = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"device: {DEV}")

DL_FEATURES = FEATURES                       # identical inputs to the models above

dl_all = labeled[labeled["REPDTE"] <= TRAIN_END]
dl_train = dl_all[dl_all["REPDTE"] <= "2010-12-31"]      # fit here
dl_val = dl_all[dl_all["REPDTE"] >= "2011-01-01"]        # early stopping here
dl_test = test_scored                                     # same scored set as above

print(f"train {len(dl_train):,} ({int(dl_train[TARGET].sum())} cases) | "
      f"val {len(dl_val):,} ({int(dl_val[TARGET].sum())}) | "
      f"test {len(dl_test):,} ({int(dl_test[TARGET].sum())})")

## Training loop

Nothing exotic: class-weighted loss so the rare cases are not drowned out, AdamW, and early
stopping on validation PR-AUC with a patience of five. The best weights by validation score are
restored before predicting, so a late epoch that overfits cannot be the one reported.

In [ ]:
def train_net(model, Xtr, ytr, Xva, yva, epochs=40, bs=4096, lr=1e-3, patience=5):
    """Fit one network. Returns it with the best-validation weights restored."""
    model = model.to(DEV)
    ratio = float((ytr == 0).sum()) / max(float((ytr == 1).sum()), 1.0)
    loss_fn = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([ratio], dtype=torch.float32, device=DEV))
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    Xva_t = torch.tensor(Xva, dtype=torch.float32, device=DEV)
    best_score, best_weights, since_improved = -1.0, None, 0

    for epoch in range(epochs):
        model.train()
        # Rows arrive sorted by bank, so consecutive rows are the same bank in
        # adjacent quarters. Without shuffling, every batch is a handful of banks
        # and the gradient is badly correlated.
        order = np.random.permutation(len(Xtr))
        for i in range(0, len(Xtr), bs):
            idx = order[i:i + bs]
            xb = torch.tensor(Xtr[idx], dtype=torch.float32, device=DEV)
            yb = torch.tensor(ytr[idx], dtype=torch.float32, device=DEV)
            opt.zero_grad()
            loss_fn(model(xb).squeeze(-1), yb).backward()
            opt.step()

        model.eval()
        with torch.no_grad():
            preds = torch.sigmoid(model(Xva_t).squeeze(-1)).cpu().numpy()
        score = average_precision_score(yva, preds)

        if score > best_score:
            best_score = score
            best_weights = {k: v.detach().clone() for k, v in model.state_dict().items()}
            since_improved = 0
        else:
            since_improved += 1
            if since_improved >= patience:
                break

    model.load_state_dict(best_weights)
    print(f"    best validation PR-AUC {best_score:.4f}")
    return model


def net_predict(model, X, bs=16384):
    model.eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(X), bs):
            xb = torch.tensor(X[i:i + bs], dtype=torch.float32, device=DEV)
            out.append(torch.sigmoid(model(xb).squeeze(-1)).cpu().numpy())
    return np.concatenate(out)


dl_results = []
dl_scores = {}          # kept so the summary can score at other budgets

## 1 · MLP on the tabular features

Two hidden layers with dropout, on the same features gradient boosting sees.

**One trap, found the hard way.** Filling blanks with the training median leaves blanks in place
for any column that is *entirely* blank in training — and blanks propagate through a network,
turning every prediction into `NaN`. The model then scores exactly 0.5 ROC-AUC and 0% recall,
which reads like a failed method rather than a broken pipeline. CET1 (`RBCT1CER`) is empty before
2015 and does exactly this. The assertion below makes it impossible to miss again.

In [ ]:
MLP_FEATURES = [c for c in DL_FEATURES if dl_train[c].notna().any()]
print(f"features: {len(MLP_FEATURES)} "
      f"(dropped as all-blank in training: {sorted(set(DL_FEATURES) - set(MLP_FEATURES))})")

fill = dl_train[MLP_FEATURES].median()


def as_matrix(df):
    return df[MLP_FEATURES].fillna(fill).to_numpy(np.float32)


Xtr = as_matrix(dl_train)
centre, spread = Xtr.mean(0), Xtr.std(0) + 1e-6
scale = lambda X: np.clip((X - centre) / spread, -10, 10)

Xtr = scale(Xtr)
Xva = scale(as_matrix(dl_val))
Xte = scale(as_matrix(dl_test))
assert not np.isnan(Xtr).any(), "blanks survived into the training matrix"

ytr = dl_train[TARGET].values.astype(np.float32)
yva = dl_val[TARGET].values.astype(np.float32)
yte = dl_test[TARGET].values.astype(np.float32)

SEEDS = [0, 1, 2]
for seed in SEEDS:
    torch.manual_seed(seed)
    np.random.seed(seed)
    mlp = nn.Sequential(
        nn.Linear(len(MLP_FEATURES), 256), nn.ReLU(), nn.Dropout(0.3),
        nn.Linear(256, 64), nn.ReLU(), nn.Dropout(0.2),
        nn.Linear(64, 1),
    )
    mlp = train_net(mlp, Xtr, ytr, Xva, yva)
    scores = net_predict(mlp, Xte)
    dl_scores.setdefault("MLP", []).append(scores)
    dl_results.append(evaluate(f"MLP (seed {seed})", scores, yte))

## 2 · GRU on eight-quarter sequences

Each row becomes a sequence: the bank's last eight quarters of twenty core measures, oldest to
newest. The lookback is built with the same self-join on `(bank, quarter − h)` used for the trend
features, so a gap in a bank's history yields a blank rather than the wrong quarter.

Two details that matter:

- **Mask channels.** Blanks are filled with the training median, and a parallel set of 0/1
  channels tells the network which values were filled. Without this it cannot distinguish a bank
  sitting at the median from a bank with no filing at all.
- **Downsampled negatives.** 300,000 negatives plus every case, the approach Petropoulos et al.
  take. Validation and test keep their true distribution — only the training fold is touched.

In [ ]:
SEQ_FEATS = ["RBCRWAJ", "RBC1AAJ", "EQV", "NPERFV", "NCLNLSR", "ORER", "NTLNLSR",
             "LNATRESR", "ROA", "ROE", "NIMY", "EEFFR", "LNLSDEPR", "CHBALR",
             "BROR", "uninsured_pct", "htm_loss_pct", "INTINCY", "INTEXPY", "ASSET"]
LOOKBACK = 8

panel["_qi"] = panel["REPDTE"].dt.year * 4 + panel["REPDTE"].dt.quarter
history = panel[["CERT", "_qi"]].copy()
for h in range(LOOKBACK):
    past = panel[["CERT", "_qi"] + SEQ_FEATS].copy()
    past["_qi"] = past["_qi"] + h
    past = past.rename(columns={c: f"{c}_t{h}" for c in SEQ_FEATS})
    history = history.merge(past, on=["CERT", "_qi"], how="left")
history.index = panel.index
panel = panel.drop(columns=["_qi"])

SEQ_COLS = [f"{c}_t{h}" for h in range(LOOKBACK) for c in SEQ_FEATS]

# ASSET spans six orders of magnitude -- log it before scaling.
for h in range(LOOKBACK):
    history[f"ASSET_t{h}"] = np.log1p(history[f"ASSET_t{h}"].clip(lower=0))

rng = np.random.RandomState(0)
case_rows = dl_train.index[dl_train[TARGET] == 1]
other_rows = rng.choice(dl_train.index[dl_train[TARGET] == 0], size=300_000, replace=False)
seq_train_idx = np.concatenate([case_rows.values, other_rows])
rng.shuffle(seq_train_idx)
print(f"training sequences: {len(seq_train_idx):,} ({len(case_rows):,} cases)")

seq_fill = history.loc[seq_train_idx, SEQ_COLS].median()


def as_sequences(idx, centre=None, spread=None):
    block = history.loc[idx, SEQ_COLS]
    missing = block.isna().to_numpy(np.float32)
    values = block.fillna(seq_fill).to_numpy(np.float32)
    if centre is not None:
        values = np.clip((values - centre) / spread, -10, 10)
    n, F, L = len(values), len(SEQ_FEATS), LOOKBACK
    # Columns run lag-major; reshape then flip so the sequence reads oldest -> newest.
    values = values.reshape(n, L, F)[:, ::-1, :].copy()
    missing = missing.reshape(n, L, F)[:, ::-1, :].copy()
    return np.concatenate([values, missing], axis=2)


raw = history.loc[seq_train_idx, SEQ_COLS].fillna(seq_fill).to_numpy(np.float32)
seq_centre, seq_spread = raw.mean(0), raw.std(0) + 1e-6

Str = as_sequences(seq_train_idx, seq_centre, seq_spread)
Sva = as_sequences(dl_val.index, seq_centre, seq_spread)
Ste = as_sequences(dl_test.index, seq_centre, seq_spread)
ytr_seq = dl_train[TARGET].loc[seq_train_idx].values.astype(np.float32)
print(f"sequence tensor: {Str.shape}  (8 quarters x {len(SEQ_FEATS)} measures + mask)")


class GRUNet(nn.Module):
    def __init__(self, n_feat, hidden=64):
        super().__init__()
        self.gru = nn.GRU(n_feat, hidden, batch_first=True)
        self.head = nn.Sequential(nn.Dropout(0.2), nn.Linear(hidden, 1))

    def forward(self, x):
        out, _ = self.gru(x)
        return self.head(out[:, -1, :])       # last step = the prediction quarter


for seed in SEEDS:
    torch.manual_seed(seed)
    np.random.seed(seed)
    gru = train_net(GRUNet(Str.shape[2]), Str, ytr_seq, Sva, yva,
                    epochs=30, bs=1024, lr=2e-3)
    scores = net_predict(gru, Ste)
    dl_scores.setdefault("GRU", []).append(scores)
    dl_results.append(evaluate(f"GRU (seed {seed})", scores, yte))

## Results

Each network is run on **three seeds**. This is not thoroughness for its own sake: an early run
of the GRU scored 0.062 and a rerun of identical code scored 0.051, purely from initialisation.
Where the spread between seeds rivals the spread between models, a single run is not a result.
Petropoulos bootstraps for the same reason.

In [ ]:
deep = pd.DataFrame(dl_results)
print(deep.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

deep["family"] = deep["model"].str.replace(r" \(seed \d\)", "", regex=True)
print("\n\nacross seeds:\n")
print(deep.groupby("family").agg(
    runs=("PR-AUC", "size"),
    roc=("ROC-AUC", "mean"),
    pr_mean=("PR-AUC", "mean"),
    pr_min=("PR-AUC", "min"),
    pr_max=("PR-AUC", "max"),
    lift=("lift", "mean"),
    recall=("recall@1%", "mean"),
).to_string(float_format=lambda v: f"{v:.4f}"))

### Verdict

**The MLP earns a place.** It posts the highest ROC-AUC of anything in this project — above the
capital-headroom benchmark — and comes second on PR-AUC behind gradient boosting. Deep learning
is competitive here, which is worth reporting given Petropoulos found the same.

**The GRU does not.** It lands below the untrained benchmark on PR-AUC, and it is the least
stable model of the set.

That negative result is the more interesting of the two. The sequence model was the natural fit
for a two-year-decline story, and it lost to models handed the same information as engineered
change columns. The most likely reading: the 4- and 8-quarter changes built in
`feature_engineering.ipynb` already contain what the sequence carries, so the GRU pays the cost
of learning the shape from scratch — on 7,738 cases — with nothing left to gain.

**And the seed variance is the methodological point.** The MLP swings roughly 18% across seeds
and the GRU 32%, both comparable to the gaps between different models. Any single-run neural
network number on this data should be treated as one draw, not as a measurement.

---

# Summary

Everything above, in one table and four conclusions.

In [ ]:
ALERT_BUDGET = 0.01     # reporting convention; see the ladder above


def caught_at(scores, y, budget=ALERT_BUDGET):
    k = int(len(scores) * budget)
    return y[np.argsort(-np.asarray(scores, float))[:k]].sum() / y.sum()


scored_by = {
    "naive (flag everything)": np.random.RandomState(0).rand(len(y_test_scored)),
    "capital headroom": headroom_scored,
    "logistic regression (alternative)":
        logistic.predict_proba(X_test_scored[LR_FEATURES])[:, 1],
    "gradient boosting (champion)": boosting.predict_proba(X_test_scored)[:, 1],
}
kind = {"naive (flag everything)": "benchmark", "capital headroom": "benchmark"}

rows = []
for name, s in scored_by.items():
    r = evaluate(name, s, y_test_scored)
    r["kind"] = kind.get(name, "method")
    rows.append(r)

for family in ["MLP", "GRU"]:
    block = deep[deep["family"] == family]
    rows.append({"kind": "method", "model": f"{family} (3-seed mean)",
                 "ROC-AUC": block["ROC-AUC"].mean(), "PR-AUC": block["PR-AUC"].mean(),
                 "lift": block["lift"].mean(),
                 "recall@1%": np.mean([caught_at(s, yte) for s in dl_scores[family]])})

summary = (pd.DataFrame(rows)[["kind", "model", "ROC-AUC", "PR-AUC", "lift", "recall@1%"]]
           .sort_values("kind", ascending=False, kind="stable"))

print(f"TEST 2017Q1-2025Q1 | {len(test_scored):,} rows | {int(y_test_scored.sum())} cases "
      f"| base rate {y_test_scored.mean():.4f}\n")
print(summary.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print("\nEvery row is measured on the same rows, the same event, and the same test set.")
print("SCOR is not included: it predicts a different event on a different population --")
print("see the context section above.")

## Five conclusions

**1. The failure has a date, and it is not a modelling failure.**
Scored one year at a time, the champion holds 0.95–0.97 ROC-AUC across 2017–2019 and recovers to
0.79 by 2024. It collapses to 0.54 in 2021 — and 2021–2023 holds 435 of the 563 test cases, 77%
of them, which is why the aggregate looks mediocre. The training years end in 2015 and are
dominated by credit-driven distress; the 2021–23 wave was rate-driven and funding-side. On the
2008 crisis, where the mechanism matches, the same model wins outright: PR-AUC 0.437 against the
benchmark's 0.297.

**2. The task is hard, and the operational record says so.**
The FDIC's own off-site system catches fewer than half the downgrades it targets, on a denser
target and with supervisory data this project does not have. Its performance is not a benchmark
here — different event, different population — but it does bound what is reasonable to claim
from public call reports alone.

**3. No method dominates, and which one wins depends on the question.**
Gradient boosting takes PR-AUC. Logistic regression catches the most cases at a fixed alert
budget. The MLP posts the best ROC-AUC of anything here. A defence claiming one method simply
won would overstate the evidence.

**4. Two inherited assumptions did not survive testing.**
The FRED macro block, joined on the belief it was literature-backed, contributes nothing
measurable — identical to four decimal places with and without, replicating Nuxoll (2003)
twenty-three years on. And the training era, which the 2010 result made look important, is flat
across sixteen start years; what matters is the number of distress cases, not the decade.

**5. What the model still cannot see is the specific thing to fix.**
Capital ratios rose into failure for SVB while its bond losses went from 0.5% to 7.6% of assets
and uninsured deposits sat at 86% against a peer median of 48%. Those funding features are now
built and they helped. They were not enough to read 2021–2023, and closing that gap — not tuning
the era, not chasing another architecture — is where the remaining work is.